# 06 — MLflow Serving

## Objectif

Tester le **serving MLflow** : charger le modèle optimisé depuis le **Model Registry**
et l'utiliser pour faire des prédictions, comme en production.

## Pourquoi le serving ?

Dans une démarche MLOps, le modèle n'est pas juste entraîné dans un notebook.
Il doit être :
1. **Versionné** dans un Model Registry (fait dans les notebooks 03/04)
2. **Chargé** depuis le Registry sans connaître les hyperparamètres
3. **Servi** via une API REST pour que d'autres applications puissent l'interroger

Ce notebook couvre les étapes 2 et 3 :
- Chargement du modèle depuis le Registry (simulation de déploiement)
- Prédiction sur de nouveaux clients
- Test du serving REST avec `mlflow models serve`

## 1. Chargement du modèle depuis le Model Registry

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Connexion au tracking server (même URI que les notebooks 04)
mlflow.set_tracking_uri('sqlite:///mlflow.db')

print('MLflow tracking URI :', mlflow.get_tracking_uri())

In [ ]:
# Lister les modèles enregistrés dans le Registry
client = mlflow.MlflowClient()

for rm in client.search_registered_models():
    print(f'Modèle : {rm.name}')
    for v in rm.latest_versions:
        print(f'  Version {v.version} | Run ID: {v.run_id} | Status: {v.status}')
    print()

### Chargement du modèle optimisé

On charge le modèle **ScoringCredit_LightGBM_Optimized** directement depuis le Registry
en utilisant son URI `models:/nom_du_modele/version`.

C'est exactement ce que ferait un service de production : il ne connaît pas les hyperparamètres,
il charge simplement le modèle sérialisé et l'utilise pour prédire.

In [ ]:
# Charger la dernière version du modèle optimisé
model_name = 'ScoringCredit_LightGBM_Optimized'
model_uri = f'models:/{model_name}/latest'

model = mlflow.pyfunc.load_model(model_uri)
print(f'Modèle chargé : {model_name}')
print(f'Type : {type(model)}')

### Récupération du seuil optimal

Le seuil de décision a été loggé comme paramètre dans le run MLflow.
On le récupère automatiquement pour ne pas le hardcoder.

In [ ]:
# Récupérer le run associé au modèle pour obtenir le seuil optimal
latest_version = client.get_registered_model(model_name).latest_versions[0]
run = client.get_run(latest_version.run_id)

seuil_optimal = float(run.data.params.get('seuil_optimal', 0.5))
print(f'Seuil optimal récupéré depuis MLflow : {seuil_optimal}')
print(f'\nMétriques du run :')
for key, value in run.data.metrics.items():
    print(f'  {key}: {value}')

## 2. Prédiction sur de nouveaux clients

On simule un scénario de production : le modèle reçoit des données de clients
et retourne une probabilité de défaut + une décision (accordé/refusé).

In [ ]:
# Charger les données et recréer le holdout set (même split que 04)
df = pd.read_csv('../data/train_preprocessed.csv')
y = df['TARGET']
X = df.drop(columns=['TARGET'])
X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Holdout set : {X_test.shape}')

In [ ]:
# Prédiction via le modèle chargé depuis le Registry
# mlflow.pyfunc renvoie directement les probabilités via predict()
predictions = model.predict(X_test)

# Si le modèle retourne des classes directement, on utilise predict_proba
# Sinon on utilise le résultat de predict
unwrapped_model = model.unwrap_python_model()
if hasattr(unwrapped_model, 'predict_proba'):
    y_proba = unwrapped_model.predict_proba(X_test)[:, 1]
else:
    # Le modèle pyfunc retourne les classes, on accède au modèle sous-jacent
    lgbm_model = model._model_impl.lgb_model if hasattr(model._model_impl, 'lgb_model') else None
    if lgbm_model is not None:
        y_proba = lgbm_model.predict_proba(X_test)[:, 1]
    else:
        y_proba = predictions  # fallback

print(f'Probabilités calculées pour {len(y_proba)} clients')
print(f'Min: {y_proba.min():.4f} | Max: {y_proba.max():.4f} | Moyenne: {y_proba.mean():.4f}')

In [ ]:
# Appliquer le seuil optimal pour la décision métier
y_pred = (y_proba >= seuil_optimal).astype(int)

# Métriques
auc = roc_auc_score(y_test, y_proba)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
cout_metier = 10 * fn + fp

print(f'=== Résultats du modèle servi depuis le Registry ===')
print(f'AUC            : {auc:.4f}')
print(f'Seuil appliqué : {seuil_optimal}')
print(f'Coût métier    : {cout_metier}  (FN={fn}, FP={fp})')
print(f'\n{classification_report(y_test, y_pred)}')

### Exemple : prédiction pour un client individuel

En production, le chargé d'études enverrait les données d'un seul client
et recevrait une réponse : probabilité de défaut + décision.

In [ ]:
# Simuler la prédiction pour un seul client
client_data = X_test.iloc[[0]]  # Premier client du holdout
proba = y_proba[0]
decision = 'REFUSÉ' if proba >= seuil_optimal else 'ACCORDÉ'

print(f'--- Prédiction pour le client ---')
print(f'Probabilité de défaut : {proba:.4f} ({proba:.2%})')
print(f'Seuil de décision     : {seuil_optimal}')
print(f'Décision              : Crédit {decision}')
print(f'TARGET réel           : {y_test.iloc[0]}')

## 3. Test du serving REST avec MLflow

MLflow permet de servir un modèle comme une **API REST** avec une seule commande.
C'est l'étape de "pré-production" demandée dans l'énoncé.

### Comment lancer le serving

Dans un terminal, exécuter :

```bash
cd notebooks
mlflow models serve -m "models:/ScoringCredit_LightGBM_Optimized/latest" --port 5001 --no-conda
```

Cela lance un serveur REST sur `http://localhost:5001` qui expose un endpoint `/invocations`.

### Comment interroger l'API

On peut envoyer des données au format JSON et recevoir les prédictions :

```bash
curl -X POST http://localhost:5001/invocations \
  -H "Content-Type: application/json" \
  -d '{"dataframe_split": {"columns": [...], "data": [[...]]}}'
```

Le code ci-dessous génère la requête et teste l'API (si le serveur est lancé).

In [ ]:
import json
import requests

# Préparer les données d'un client au format attendu par MLflow serving
sample = X_test.iloc[:3]  # 3 clients pour le test
payload = {
    'dataframe_split': sample.to_dict(orient='split')
}

# Afficher la commande pour lancer le serving
print('=== Pour lancer le serving, exécuter dans un terminal ===')
print(f'cd notebooks')
print(f'mlflow models serve -m "models:/{model_name}/latest" --port 5001 --no-conda')
print()

# Tester l'API si le serveur est lancé
SERVING_URL = 'http://localhost:5001/invocations'

try:
    response = requests.post(
        SERVING_URL,
        headers={'Content-Type': 'application/json'},
        data=json.dumps(payload),
        timeout=10
    )
    if response.status_code == 200:
        predictions = response.json()
        print(f'Réponse du serveur (status {response.status_code}) :')
        print(f'Prédictions : {predictions}')
    else:
        print(f'Erreur serveur (status {response.status_code}) : {response.text}')
except requests.exceptions.ConnectionError:
    print('Le serveur MLflow serving n\'est pas lancé.')
    print('Lancez-le dans un terminal avec la commande ci-dessus, puis relancez cette cellule.')
    print()
    print('Payload qui serait envoyé (aperçu) :')
    print(json.dumps({'dataframe_split': {'columns': list(sample.columns[:5]) + ['...'], 'data': '[[...]]'}}, indent=2))

## Conclusion

### Récapitulatif de la démarche MLOps mise en place

| Étape MLOps | Notebook | Outil |
|---|---|---|
| **Tracking** des expérimentations | 03, 04 | `mlflow.log_param()`, `mlflow.log_metric()` |
| **UI MLflow** pour visualiser les résultats | — | `mlflow ui` → http://localhost:5000 |
| **Model Registry** pour versionner les modèles | 03, 04 | `mlflow.sklearn.log_model(registered_model_name=...)` |
| **Serving** pour tester la pré-production | 06 | `mlflow models serve` + `mlflow.pyfunc.load_model()` |

### Ce que le serving apporte

- **Découplage** : l'application de scoring n'a pas besoin de connaître LightGBM, ses hyperparamètres,
  ni le code d'entraînement. Elle interroge une API REST standard
- **Versionnement** : on peut déployer une nouvelle version du modèle dans le Registry
  et le serving pointe automatiquement dessus
- **Reproductibilité** : chaque version du modèle est liée à un run MLflow
  (hyperparamètres, métriques, données d'entraînement)

### Architecture cible en production

```
Client → API Gateway → MLflow Serving (port 5001) → Prédiction
                                ↓
                    Model Registry (modèle versionné)
                                ↓
                    Tracking Server (métriques, logs)
```